# Rounded bowtie waveguide

This notebook introduces NumBAT's `rounded_bowtie` structure: two symmetric tapered inclusions with rounded inner tips, rounded outer transitions, and a circular outer domain. It builds the Gmsh geometry, displays the mesh and refractive-index profile, then solves and visualizes the first two electromagnetic modes at 1550 nm.

All geometric dimensions below are in **nm**. `alpha_tip` is a half-angle in **radians**.

## Imports and notebook helpers

The notebook locates NumBAT relative to this file, so it can be started from the repository root or from the tutorial directory. The helper below displays PNG files written by NumBAT.

In [30]:
%matplotlib inline
%load_ext autoreload
%autoreload 2



import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from plotting.ipythontools import img_single, img_pair, img_triple

repository_root = Path.cwd()
if not (repository_root / "backend").is_dir():
    repository_root = Path.cwd().parents[1]
sys.path.append(str(repository_root / "backend"))

import materials  # noqa: E402
import numbat  # noqa: E402


def display_pngs(directory, pattern="*.png"):
    """Display NumBAT plot files created in directory."""
    for filename in sorted(Path(directory).glob(pattern)):
        display(Image(filename=str(filename)))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Geometry and materials

The gap is measured between the nearest points of the two rounded tips.

The key parameters are:
- `bowlen_l`: the distance from the origin to each inclusion's circular outer boundary.
- `gap`: the distance between the nearest points of the two rounded tips.


This example uses two arsenic trisulfide (`As₂S₃`) inclusions in silica so both optical and elastic fields are bound. 

Set `one_wedge = True` to replace the left inclusion with the background material.

In [ ]:
lambda_nm = 1550
domain_x = 3000
domain_y = 3000

rad_tip = 20
alpha_tip = np.deg2rad(20)
gap_g = 30
bowlen_l = 220
rad_corner = 20
rad_wg = 500

num_modes_EM = 20
#mode_indices = [0, 1]
emout_m = range(5)
acout_m=range(50)
prefix = "tt_20_rounded_bowtie"

one_wedge = False
silica = materials.make_material("SiO2_2023_Steel")
chalcogenide = materials.make_material("As2S3_2023_Steel")
material_bkg = silica
material_a = chalcogenide
material_b = material_bkg if one_wedge else material_a

## Build the rounded bowtie structure

Creating the structure loads the built-in Gmsh template, substitutes the parameters above, checks the geometry constraints, invokes Gmsh, and assigns the physical surfaces to the three selected materials.

In [14]:
nbapp = numbat.NumBATApp(prefix)
wguide = nbapp.make_structure(
    "rounded_bowtie",
    domain_x,
    domain_y,
    rad_tip=rad_tip,
    rad_wg=rad_wg,
    alpha_tip=alpha_tip,
    gap_g=gap_g,
    bowlen_l=bowlen_l,
    rad_corner=rad_corner,
    material_bkg=material_bkg,
    material_a=material_a,
    material_b=material_b,
    lc_bkg=0.02,
    lc_refine_1=5.0,
    lc_refine_2=5.0,
)

 Calling Gmsh: 

Let's look at the generated structure from mesh and physical standpoints.

In [17]:
fn_mesh_plot  = wguide.plot_mesh(prefix)
img_single(fn_mesh_plot, 'Design and mesh', 400)


### Physical geometry
And now 2D plots and cut lines of some of the physical parameters.

First the **refractive index**

In [23]:
index_plotter = wguide.get_structure_plotter_refractive_index()
[fn_ind_2D] = index_plotter.make_plot_2D(prefix)
[fn_ind_1D] = index_plotter.make_plot_1D_cut(prefix, "y", 0)

img_pair(fn_ind_2D, fn_ind_1D, 'Refractive index', 400)

    initial mesh has: 6017 nodes, 2980 elements. 


 The final EM sim mesh has 6017 nodes, 2980 elements and 3 element types (materials).
 There are 3 active materials:
  SiO2_2023_Steel,     n = 1.45000+0.00000j, mat. index = 1.
  As2S3_2023_Steel,    n = 2.45000+0.00000j, mat. index = 2.
  As2S3_2023_Steel,    n = 2.45000+0.00000j, mat. index = 3.


And now the **elastic phase velocities**

In [ ]:
el_vphase_plotter = wguide.get_structure_plotter_acoustic_velocity()
l_fn_2D = el_vphase_plotter.make_plot_2D(prefix)
[fn_evlph_1D] = el_vphase_plotter.make_plot_1D_cut(prefix, "y", 0)

img_triple(l_fn_2D[0], l_fn_2D[1], fn_evlph_1D, 'Elastic phase velocities', 400)

    initial mesh has: 6017 nodes, 2980 elements. 


 The final EM sim mesh has 6017 nodes, 2980 elements and 3 element types (materials).
 There are 3 active materials:
  SiO2_2023_Steel,     n = 1.45000+0.00000j, mat. index = 1.
  As2S3_2023_Steel,    n = 2.45000+0.00000j, mat. index = 2.
  As2S3_2023_Steel,    n = 2.45000+0.00000j, mat. index = 3.
['tt_20_rounded_bowtie-elastic_velocity_v0.png', 'tt_20_rounded_bowtie-elastic_velocity_v1.png', 'tt_20_rounded_bowtie-elastic_velocity_v2.png']


## Solve electromagnetic modes

Let's find the lowest few EM modes at the default wavelength of 1550 nm.

In [ ]:
n_eff_estimate = material_a.refindex_n - 0.1
sim_EM = wguide.calc_EM_modes(num_modes_EM, lambda_nm, n_eff=n_eff_estimate)

v_neff = sim_EM.neff_all()[]
for mode_index, n_eff in enumerate(sim_EM.neff_all()):
    print(f"mode {mode_index:2d}: n_eff = {n_eff:.6f}")

    initial mesh has: 6017 nodes, 2980 elements. 


 The final EM sim mesh has 6017 nodes, 2980 elements and 3 element types (materials).
 There are 3 active materials:
  SiO2_2023_Steel,     n = 1.45000+0.00000j, mat. index = 1.
  As2S3_2023_Steel,    n = 2.45000+0.00000j, mat. index = 2.
  As2S3_2023_Steel,    n = 2.45000+0.00000j, mat. index = 3.


Calculating EM modes:
 Boundary conditions: Dirichlet

 -----------------------------------------------
 EM FEM: 
  Calculating EM mode powers...
using curvilinear elements
   - assembling linear system:
           2980 mesh elements
           6017 mesh nodes
          35929 linear equations (cscmat%n_dof)
        1226125 nonzero elements  (cscmat%n_nonz)
          0.095 % sparsity
             32 MB est. working memory 

       cpu time = .29 secs, wall time = .29 secs

  - solving linear system: 

      solving eigensystem
         cpu time = 258.29 secs, wall time = 8.55 secs

      assembling modes
         cpu time = 2.85 secs, wall

## First two 2D electromagnetic modes

These plots show the electric-field components and magnitude for modes `0` and `1`. The wire-frame border is the same rounded-bowtie geometry used by Gmsh.

In [ ]:
sim_EM.plot_modes(mode_indices=mode_indices, hide_vector_field=True)
display_pngs(nbapp.outdir_fields_path(prefix), "E*_mode_0[01].png")

## Horizontal field cuts at y = 0

The horizontal cut passes through the two rounded tips and the central gap. Comparing the first two modes along this line makes their symmetry and gap confinement easy to see.

In [ ]:
sim_EM.plot_modes_1D("y", 0, mode_indices=mode_indices)
display_pngs(nbapp.outdir_fields_path(prefix), "E*_mode_0[01]_ycut*.png")

## Next steps

NumBAT stores field plots under `jup_20_rounded_bowtie-fields/` and structure plots in the notebook's output directory. Change the geometry cell to study the tip radius, gap, half-angle, inclusion radius, or outer-corner radius. Decrease `lc_bkg` or increase the refinement factors for a finer mesh, then repeat the structure and EM-mode cells.